In [0]:
%run ../control_framework/__init__ 

In [0]:
dbutils.widgets.text("env", "dev", "env")
# dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "finnhub_daily_ingestion", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")
dbutils.widgets.text("run_type", "load", "run type")
dbutils.widgets.text("source", "finnhub", "Source")
dbutils.widgets.text("target", "m101_finnhub", "Target")
dbutils.widgets.text("execution_type", "ingestion", "execution type")

In [0]:
env = dbutils.widgets.get("env")
# param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
run_type = dbutils.widgets.get("run_type")
source = dbutils.widgets.get("source")
target = dbutils.widgets.get("target")
execution_type = dbutils.widgets.get("execution_type")
print(f"env: {env}")
# print(f"param_name: {param_name}")
print(f"process_name: {process_name}")
print(f"process_date: {process_date}")
print(f"run_type: {run_type}")
print(f"source: {source}")
print(f"target: {target}")
print(f"execution_type: {execution_type}")

In [0]:
try:
    comment = register_tgt(target, execution_type)
    insert_log(target, process_name, "source_register","source_register","source_register","source_register",process_date, 'success',comment)
except Exception as e:
    error_msg = str(e).replace("'", "''")
    insert_log(target, process_name, "source_register","source_register","source_register","source_register",process_date, 'failed', error_msg)
    print(f"Error in register_source: {e}")
    dbutils.notebook.exit(f"Error in register_source: {e}")

In [0]:
src_schema1 = f_get_src_schema(source)
src_tbl1 = ''
src_schema_master = f_get_src_schema('master')
src_master_tb1 = 's_&_p_500_dev'
target_schema = f_get_tgt_schema(target)
target_tbl = 'finnhubb_tb1_ts'
print(src_schema1)
print(src_tbl1)
print(src_schema_master)
print(src_master_tb1)
print(target_schema)
print(target_tbl)


In [0]:
src_tbl_list = [{"source":source,"source_schema":src_schema1,"source_table": src_tbl1},
                {"source":'master',"source_schema":src_schema_master,"source_table": src_master_tb1}
                ]
tgt_tbl_list = [{"target":target,"target_schema":target_schema,"target_table": target_tbl,'holiday_ind': 'N','weekend_ind': 'N','calender':'','frequency': 'daily'}]

In [0]:
# {'c': 333.74, 'd': 0.48, 'dp': 0.144, 'h': 334.99, 'l': 329.0006, 'o': 331.98, 'pc': 333.26, 't': 1784318400}

In [0]:
src_tgt_mapping_list = [
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'c','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'c','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'d','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'d','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'dp','source_column_datatype':'decimal(20,3)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'dp','target_column_datatype':'decimal(20,3)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'h','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'h','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'l','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'l','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'o','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'o','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'pc','source_column_datatype':'decimal(20,2)','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'pc','target_column_datatype':'decimal(20,2)'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'t','source_column_datatype':'bigint','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'t','target_column_datatype':'bigint'},
    {'source_schema':src_schema_master,'source_table':src_master_tb1,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'symbol','target_column_datatype':'string'}
]

In [0]:
if run_type == 'load_metadata':
    status = run_load_metadata(src_tbl_list,tgt_tbl_list,target,process_name,src_tgt_mapping_list,process_date)
    dbutils.notebook.exit(status)


In [0]:
if run_type == 'full':
    create_tgt_tbl_log = create_tgt_tbl(tgt_tbl_list)
    if  create_tgt_tbl_log == f"Table {target_schema}.{target_tbl} created successfully":
        process_status = 'success'
    else:
        process_status = 'failed'
    log = str(create_tgt_tbl_log).replace("'", "''")
    insert_log (target, process_name, 'create_tgt_tbl','create_tgt_tbl', target_schema, target_tbl, process_date,process_status, log)

In [0]:
%sql
select * from ni_m101_finnhub_dev.finnhubb_tb1_ts

In [0]:
%sql
select * from metadata_dev.audit_log_dev